# Unit regions by reading (b): decoding and dPCA claims, Boran et al. (2019), dandiset 000574

Third notebook of the entry. The claims file (`neurons-recorded` criterion, amended 2026-10-30)
takes the region of a unit from the tip contact of its depth electrode, (a) by the anatomical
label updated after visual inspection of MRI, (b) by the Brainnetome label, for every claim
that groups neurons by region. The first notebook runs reading (a) for every claim and reading
(b) for the selection claims; this notebook reruns the decoding and dPCA claims under reading
(b), which would take the first notebook over its time limit. Dataset version 0.250815.1108
(see the first notebook's header for why).

Reading (b) differs from (a) only for units on the four tip contacts whose label the G-Node
deposit (doi:10.12751/g-node.d76994, commit 0b27b34) flags as entered by hand: their
Brainnetome label is in neither deposit, so they are left out (link claim `electrode-labels-gin`
in the first notebook). Each section reruns the main reading of its claim with the same code
as the first notebook, with its own generator; the outcomes are judged in the first notebook
and the attempt record, which say whether reading (b) changes them.

| Claim | Run here |
|---|---|
| fig4e-maintenance-decodes-workload | yes, reading (b) |
| fig4d-encoding-decodes-correct | yes, reading (b) |
| fig4e-workload-decoding-leave-one-subject-out | yes, reading (b) |
| fig4e-maintenance-neurons-carry-workload | yes, reading (b) |
| fig4e-no-decoding-6-vs-8 | yes, reading (b) |
| fig4f-retrieval-decodes-response | yes, reading (b) |
| fig4a-dpca-workload-separation | yes, reading (b) |
| fig4b-speed-lowest-maintenance | yes, reading (b) |
| fig4c-distance-largest-maintenance | yes, reading (b) |
| figs3d-hippocampus-loadings | yes, reading (b) |

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import sternberg as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

In [ ]:
A = S.assets()
D = S.load_all(A, workers=16)
GE = S.gin_electrodes()
print(len(A), "assets read at version", S.DANDISET_VERSION, "; G-Node contacts", len(GE))

In [ ]:
MAN = GE[GE.manual_entry]
MANUAL_TIPS = set(zip(MAN.subject, MAN.label))
print("tip contacts flagged as manually entered in the G-Node deposit:")
print(MAN[["subject", "label", "gin_location"]].to_string(index=False))
ds_loc = {(d["subject"], l): loc for d in D for l, loc in zip(d["electrodes"].label, d["electrodes"].location)}
print("their location in the dandiset:", {f"{s} {l}": ds_loc[(s, l)] for s, l in sorted(MANUAL_TIPS)})

In [ ]:
EDGES = np.round(np.arange(-6.0, 2.0 + 1e-9, 0.02), 4)
units = []
for d in D:
    tr = d["trials"]
    lab = dict(zip(d["electrodes"].label, d["electrodes"].location))
    for u in d["units"]:
        al = S.aligned_spikes(u["spikes"], tr.probe.to_numpy())
        reg_a, reg_b, _ = S.tip_regions(d["subject"], u["bundle"], lab, MANUAL_TIPS)
        units.append({"subject": d["subject"], "session": d["session"], "sess": d["subject"] + "-" + d["session"],
                      "unit": u["unit"], "bundle": u["bundle"], "region_bundle": u["region"],
                      "region_atlas": reg_a, "region_atlas_b": reg_b,
                      "aligned": al, "bins": S.binned(al, EDGES).astype(np.int16),
                      "correct": tr.correct.to_numpy(), "set_size": tr.set_size.to_numpy(),
                      "is_in": tr.is_in.to_numpy(), "rt": tr.rt.to_numpy(),
                      "spikes": u["spikes"], "obs": u["obs"],
                      "wm": u["waveform_mean"], "wsd": u["waveform_sd"]})
UN = pd.DataFrame([{k: v for k, v in u.items() if k in ("subject", "session", "sess", "unit", "bundle", "region_bundle", "region_atlas", "region_atlas_b")} for u in units])
REG = ["hippocampus", "entorhinal", "amygdala"]
print(len(units), "units; regions (a)", UN.region_atlas.value_counts().to_dict(), "; (b)", UN.region_atlas_b.value_counts().to_dict())


def win_counts(u, window):
    i0 = int(round((window[0] + 6) / 0.02))
    i1 = int(round((window[1] + 6) / 0.02))
    return u["bins"][:, i0:i1].sum(1).astype(float)


def win_rate(u, window):
    return win_counts(u, window) / (window[1] - window[0])


def sliding(u, width, lo, hi, step=0.02):
    """Rate in windows of `width` s centred on times lo..hi at 20 ms steps (trials x times)."""
    w = int(round(width / 0.02))
    cs = np.concatenate([np.zeros((u["bins"].shape[0], 1)), np.cumsum(u["bins"], 1)], 1)
    centres = np.round(np.arange(lo, hi + 1e-9, step), 4)
    out = []
    for c in centres:
        a = int(round((c - width / 2 + 6) / 0.02))
        a = min(max(a, 0), u["bins"].shape[1] - w)
        out.append((cs[:, a + w] - cs[:, a]) / width)
    return np.array(out).T, centres

In [ ]:
print("units by region, reading (a) against reading (b):")
print(pd.crosstab(UN.region_atlas, UN.region_atlas_b, margins=True))
print("units left out under (b), by subject and bundle:", UN[UN.region_atlas_b == "no Brainnetome label"].groupby(["subject", "bundle"]).size().to_dict())

## Maintenance neurons

The selection of fig2b (paired and unpaired permutation t test, maintenance against fixation,
correct trials), run with the first notebook's code and generator so that the same neurons are
selected. It does not depend on regions; `fig4e-maintenance-neurons-carry-workload` uses it.

In [ ]:
rng = np.random.default_rng(201)
R = 1000
sel = {"paired": [], "unpaired": []}
null_sel = {"paired": [], "unpaired": []}
p_mf = {"paired": [], "unpaired": []}
for u in units:
    c = u["correct"]
    m, f = win_rate(u, S.MAINT)[c], win_rate(u, S.FIX)[c]
    n = len(m)
    d = m - f
    # paired: sign flips
    t_ref = S.t_paired_rows(rng.choice([-1.0, 1.0], (2000, n)) * d)
    t0 = S.t_paired_rows(d[None])[0]
    p0 = S.perm_p(t0, t_ref)
    tr_ = S.t_paired_rows(rng.choice([-1.0, 1.0], (R, n)) * d)
    sel["paired"].append(bool(p0 < 0.05 and t0 > 0))
    null_sel["paired"].append((S.perm_p(tr_, t_ref) < 0.05) & (tr_ > 0))
    p_mf["paired"].append(p0)
    # unpaired: values of the two periods pooled
    x = np.concatenate([m, f])[:, None]
    t_ref = S.t_groups(x, S.random_masks(n, 2 * n, 2000, rng))[:, 0]
    g0 = np.r_[np.ones(n, bool), np.zeros(n, bool)]
    t0 = S.t_groups(x, g0[None])[0, 0]
    p0 = S.perm_p(t0, t_ref)
    sw = rng.random((R, n)) < 0.5
    tr_ = S.t_groups(x, np.concatenate([~sw, sw], 1))[:, 0]
    sel["unpaired"].append(bool(p0 < 0.05 and t0 > 0))
    null_sel["unpaired"].append((S.perm_p(tr_, t_ref) < 0.05) & (tr_ > 0))
    p_mf["unpaired"].append(p0)
for k in sel:
    UN["maint_" + k] = sel[k]
    null_sel[k] = np.array(null_sel[k])

print("maintenance neurons (paired, unpaired) under (b), by region:")
print(UN.groupby("region_atlas_b")[["maint_paired", "maint_unpaired"]].sum())

## Population decoding, reading (b)

As in the first notebook: 100 runs and 250 null runs, P times 12 (Bonferroni).

In [ ]:
CENTERS = {"encoding": np.arange(-4.75, -3.25 + 1e-9, 0.25), "maintenance": np.arange(-2.75, -0.25 + 1e-9, 0.25)}


def feats_for(u, centres, kind="boxcar"):
    if kind == "boxcar":
        X, _ = None, None
        cs = np.concatenate([np.zeros((u["bins"].shape[0], 1)), np.cumsum(u["bins"], 1)], 1)
        out = []
        for c in centres:
            a = int(round((c - 0.75 + 6) / 0.02))
            b = int(round((c + 0.75 + 6) / 0.02))
            a, b = max(a, 0), min(b, u["bins"].shape[1])
            out.append((cs[:, b] - cs[:, a]) / ((b - a) * 0.02))
        return np.array(out).T
    t = np.round(np.arange(-6.0, 2.0, 0.02) + 0.01, 4)
    out = []
    for c in centres:
        w = np.exp(-0.5 * ((t - c) / 0.25) ** 2) * (np.abs(t - c) <= 0.75)
        out.append(u["bins"] @ (w / (w.sum() * 0.02)))
    return np.array(out).T


def build(pop_idx, label_fn, need, centres, kind="boxcar"):
    """Features and labels of the neurons in pop_idx that have at least need[c] trials per class."""
    F, L, used = [], [], []
    for i in pop_idx:
        u = units[i]
        y, keep = label_fn(u)
        if np.sum(y[keep] == 0) < need[0] or np.sum(y[keep] == 1) < need[1]:
            continue
        F.append(feats_for(u, centres, kind)[keep])
        L.append(y[keep].astype(int))
        used.append(i)
    return F, L, used


def summarize(acc, null, n_tests=12):
    a_med = np.median(acc.mean(1))
    b_med = np.median(acc)
    pa = (np.sum(null.mean(1) >= a_med) + 1) / (len(null) + 1)
    pb = (np.sum(null.ravel() >= b_med) + 1) / (null.size + 1)
    return {"median_a": 100 * a_med, "median_b": 100 * b_med, "null_median": 100 * np.median(null.mean(1)),
            "null_95": 100 * np.percentile(null.mean(1), 95), "P_a": pa, "P_a_bonf": min(1, pa * n_tests),
            "P_b": pb, "P_b_bonf": min(1, pb * n_tests)}


def region_idx(r, regcol="region_atlas_b", extra=None):
    m = (UN[regcol] == r) if r != "all" else UN[regcol].isin(REG)
    if extra is not None:
        m &= extra
    return list(np.where(m)[0])

### fig4e-maintenance-decodes-workload

> During maintenance, workload (set size 4 against 6 and 8) could be decoded from hippocampal
> neurons with a median accuracy of 80%, but not from neurons of the other areas.

Main reading (boxcar, z-scored, 10 trials per class). The entorhinal and amygdala neurons are
the same under both readings; their decoding is in the first notebook.

In [ ]:
rng = np.random.default_rng(411)
NR, NN = 100, 250
lab_load = lambda u: ((u["set_size"] >= 6).astype(int), u["correct"])
for r in REG:
    print(f"{r}: neurons under (a) {int((UN.region_atlas == r).sum())}, under (b) {int((UN.region_atlas_b == r).sum())}")
F, L, used = build(region_idx("hippocampus"), lab_load, {0: 10, 1: 10}, CENTERS["maintenance"])
acc_h = S.decode(F, L, 10, NR, rng)
null_h = S.decode(F, L, 10, NN, rng, scramble=True)
all_h = summarize(acc_h, null_h)
print(f"hippocampus (b): {len(used)} neurons;", {k: round(v, 4) for k, v in all_h.items()})

**Result: Same outcome as reading (a).** Under reading (b), 615 hippocampal neurons decode workload during maintenance at a median of 81.4% (paper 80 +/- 8), above the scrambled null (P = 0.004, Bonferroni 0.048, the smallest this run allows). The entorhinal and amygdala neurons are the same 421 and 398 as under reading (a).

### fig4d-encoding-decodes-correct

> During encoding of high-workload trials (set sizes 6 and 8), hippocampal neurons, but not
> entorhinal or amygdala neurons, predicted whether the subject would later respond correctly,
> with a median decoding accuracy of 80%.

In [ ]:
rng = np.random.default_rng(412)
lab_corr = lambda u: (u["correct"].astype(int), u["set_size"] >= 6)
F, L, used = build(region_idx("hippocampus"), lab_corr, {0: 5, 1: 5}, CENTERS["encoding"])
acc = S.decode(F, L, 5, NR, rng)
null = S.decode(F, L, 5, NN, rng, scramble=True)
print(f"hippocampus (b): {len(used)} neurons from {UN.sess.iloc[used].nunique()} sessions;",
      {k: round(v, 4) for k, v in summarize(acc, null).items()})

**Result: Same outcome as reading (a).** Under reading (b), hippocampal neurons (201 from 10 sessions) decode later correctness at 51.4% (paper 80 +/- 8), at the level of the scrambled null (P = 0.46).

### fig4e-workload-decoding-leave-one-subject-out

> Excluding any one subject changed the hippocampal workload decoding accuracy to a value in
> the range 71 to 81%.

In [ ]:
rng = np.random.default_rng(413)
loso = {}
for sid in sorted(UN.subject.unique()):
    F, L, used = build(region_idx("hippocampus", extra=(UN.subject != sid)), lab_load, {0: 10, 1: 10}, CENTERS["maintenance"])
    acc = S.decode(F, L, 10, NR, rng)
    loso[sid] = (100 * np.median(acc.mean(1)), 100 * np.median(acc), len(used))
L_ = pd.DataFrame(loso, index=["median_a", "median_b", "neurons"]).T
print(L_.round(2).to_string())
print("range, reading (a): %.1f to %.1f%%; reading (b): %.1f to %.1f%%" % (L_.median_a.min(), L_.median_a.max(), L_.median_b.min(), L_.median_b.max()))

**Result: Same outcome as reading (a).** Leaving out the hippocampal units of one subject gives medians from 71.4 to 81.8%, inside the widened range 63.9 to 89.1%.

### fig4e-maintenance-neurons-carry-workload

> Workload decoding from hippocampal maintenance neurons alone was little different from
> decoding from all hippocampal neurons, while excluding the maintenance neurons reduced the
> accuracy to 71%.

In [ ]:
rng = np.random.default_rng(414)
print("all hippocampal neurons: median (a) %.1f%%, (b) %.1f%%" % (all_h["median_a"], all_h["median_b"]))
for k in ["paired", "unpaired"]:
    for name, extra in [("(i) maintenance neurons only", UN["maint_" + k]), ("(ii) without maintenance neurons", ~UN["maint_" + k])]:
        F, L, used = build(region_idx("hippocampus", extra=extra), lab_load, {0: 10, 1: 10}, CENTERS["maintenance"])
        acc = S.decode(F, L, 10, NR, rng)
        null = S.decode(F, L, 10, NN, rng, scramble=True)
        sm = summarize(acc, null)
        print(f"fig2b {k}, {name}: {len(used)} neurons;", {kk: round(v, 4) for kk, v in sm.items()},
              f"; difference from all, (a) {sm['median_a'] - all_h['median_a']:+.1f} points")

**Result: Same outcome as reading (a).** All 615 hippocampal neurons decode workload at 81.4%; the 126 maintenance neurons of the paired selection alone give 80.9% (0.5 points lower, P = 0.004) and the other 489 give 72.3% (paper 71 +/- 7.1). The unpaired selection gives 84.1% and 70.2%.

### fig4e-no-decoding-6-vs-8

> The decoder did not distinguish trials of set size 6 from set size 8.

In [ ]:
rng = np.random.default_rng(415)
lab_68 = lambda u: ((u["set_size"] == 8).astype(int), u["correct"] & (u["set_size"] >= 6))
F, L, used = build(region_idx("hippocampus"), lab_68, {0: 10, 1: 10}, CENTERS["maintenance"])
acc = S.decode(F, L, 10, NR, rng)
null = S.decode(F, L, 10, NN, rng, scramble=True)
print(f"hippocampus, set size 6 against 8: {len(used)} neurons;", {k: round(v, 4) for k, v in summarize(acc, null, n_tests=1).items()})

**Result: Same outcome as reading (a).** Set size 6 against 8 is decoded from 513 hippocampal neurons at 56.8%, not above the scrambled null (P = 0.18).

### fig4f-retrieval-decodes-response

> During retrieval (500 ms before the button press), neurons in all recorded areas predicted
> the IN or OUT response; with all neurons the median decoding accuracy reached 90%.

Hippocampus and all neurons (the other areas are unchanged under reading (b)).
The all-neuron population is also rerun with three other generators under both readings.


In [ ]:
rng = np.random.default_rng(416)


def build_retrieval(pop_idx):
    F, L, used = [], [], []
    for i in pop_idx:
        u = units[i]
        keep = u["correct"] & (u["rt"] <= 2.0)
        y = u["is_in"].astype(int)
        if np.sum(y[keep] == 0) < 10 or np.sum(y[keep] == 1) < 10:
            continue
        cnt = np.array([np.sum((s >= r - 0.5) & (s < r)) for s, r in zip(u["aligned"], u["rt"])]) / 0.5
        F.append(cnt[keep][:, None])
        L.append(y[keep])
        used.append(i)
    return F, L, used


for regcol in ["region_atlas_b"]:
    for r in ["hippocampus", "all"]:
        F, L, used = build_retrieval(region_idx(r, regcol))
        acc = S.decode(F, L, 10, NR, rng)
        null = S.decode(F, L, 10, NN, rng, scramble=True)
        print(f"{regcol} {r}: {len(used)} neurons;", {k: round(v, 4) for k, v in summarize(acc, null).items()})
# The all-neuron P sits near the smallest value 250 null runs allow; the same decoder with three
# other generators, under both readings, shows how stable its significance is.
for regcol in ["region_atlas", "region_atlas_b"]:
    F, L, used = build_retrieval(region_idx("all", regcol))
    for seed in (421, 422, 423):
        rng_s = np.random.default_rng(seed)
        acc = S.decode(F, L, 10, NR, rng_s)
        null = S.decode(F, L, 10, NN, rng_s, scramble=True)
        sm = summarize(acc, null)
        print(f"repeat, {regcol} all, generator {seed}: {len(used)} neurons; median (a) {sm['median_a']:.1f}%, "
              f"P {sm['P_a']:.4f}, corrected {sm['P_a_bonf']:.4f}")


**Result: Same outcome as reading (a), not reproduced.** Under reading (b), hippocampal neurons decode IN against OUT at 70%, not above their null (P = 0.096, corrected 1). All 1434 neurons reach 80% (paper 90 +/- 9, outside the tolerance) with P = 0.004, corrected 0.048, the smallest 250 null runs allow. With three other generators the corrected P of the all-neuron population is 0.10 to 0.38 under reading (b) and 0.10 to 0.62 under reading (a), so its significance is not stable under either reading. The entorhinal and amygdala populations are those of reading (a), where neither is significant.

## Population state space (dPCA), reading (b)

As in the first notebook, on the neurons of the three areas under reading (b).

### fig4a-dpca-workload-separation

> The first three demixed principal components of the population firing rates explained 44.7%
> of the variance and separated set sizes 4, 6 and 8, with three angles of 120 degrees in the
> dPC2-dPC3 plane; the first 15 components explained 79%.

In [ ]:
rng = np.random.default_rng(511)
TT = np.round(np.arange(-6.0, 2.0, 0.02) + 0.01, 4)
sig_ = (1.0 / 6) / 0.02
kern = np.exp(-0.5 * (np.arange(-25, 26) / sig_) ** 2)
kern /= kern.sum()
pop = [i for i in range(len(units)) if UN.region_atlas_b[i] in REG]


SM = []     # per neuron: smoothed rates of correct trials, their set sizes, fixation mean and SDs
for i in pop:
    u = units[i]
    c = u["correct"]
    r = np.apply_along_axis(lambda x: np.convolve(x, kern, mode="same"), 1, u["bins"][c].astype(float)) / 0.02
    fix_tr = win_rate(u, S.FIX)[c]
    SM.append((r, u["set_size"][c], fix_tr.mean(), fix_tr.std(ddof=1), r.mean(0).std()))


def psth_z(k, mode, ss=None):
    r, s, m, sd_a, sd_b = SM[k]
    s = s if ss is None else ss
    sd = sd_a if mode == "a" else sd_b
    return (np.array([r[s == v].mean(0) for v in (4, 6, 8)]) - m) / (sd if sd > 0 else 1.0)


XZ = {mode: np.array([psth_z(k, mode) for k in range(len(pop))]) for mode in ("a", "b")}
maint_t = (TT >= -3) & (TT < 0)


def angles(comps):
    P = np.array([[comps[1]["z"][s][maint_t].mean(), comps[2]["z"][s][maint_t].mean()] for s in range(3)])
    v = P - P.mean(0)
    ang = np.degrees(np.arctan2(v[:, 1], v[:, 0]))
    order = np.argsort(ang)
    a = np.diff(np.r_[ang[order], ang[order][0] + 360])
    return np.sort(a), P


DP = {}
for mode in ("a", "b"):
    for lam in (0.0, 1e-2):
        comps, Xf, tot = S.dpca(XZ[mode], 15, lam)
        DP[(mode, lam)] = comps
        ev3, ev15 = S.cumulative_ev(comps, Xf, tot, 3), S.cumulative_ev(comps, Xf, tot, 15)
        ang, P = angles(comps)
        print(f"z-score ({mode}), lambda {lam}: first 3 dPCs {ev3:.1f}% (sum of single {sum(c['ev'] for c in comps[:3]):.1f}%), "
              f"first 15 {ev15:.1f}%; marginalizations of dPC1-3 {[c['marg'] for c in comps[:3]]}; "
              f"angles in the dPC2-dPC3 plane {np.round(ang, 1).tolist()}")
# separation against label-shuffled dPCA (main: z-score a, no regularization)
def pair_dists(comps):
    Z = np.array([[c["z"][s][maint_t] for c in comps[:15]] for s in range(3)])   # set x comp x time
    return [float(np.mean(np.sqrt(((Z[i] - Z[j]) ** 2).sum(0)))) for i, j in [(0, 1), (0, 2), (1, 2)]]


obs_d = pair_dists(DP[("a", 0.0)])
null_d = []
for _ in range(40):
    Xs = [psth_z(k, "a", rng.permutation(SM[k][1])) for k in range(len(pop))]
    null_d.append(pair_dists(S.dpca(np.array(Xs), 15, 0.0)[0]))
null_d = np.array(null_d)
print("maintenance distance between set sizes (4-6, 4-8, 6-8):", np.round(obs_d, 3).tolist(),
      "; label-shuffled 95th percentiles:", np.round(np.percentile(null_d, 95, axis=0), 3).tolist(),
      "; P:", [(np.sum(null_d[:, k] >= obs_d[k]) + 1) / 41 for k in range(3)])

**Result: Same outcome as reading (a).** The first three dPCs explain 32.7% (paper 44.7 +/- 4.47) and the first 15 explain 64.2% (79 +/- 7.9) with the fixation-SD z-score; the three set sizes lie on a line in the dPC2-dPC3 plane (angles 0, 180 and 180 degrees, not 120), and during maintenance 6 and 8 are not farther apart than in label-shuffled runs (P = 0.73).

### fig4b-speed-lowest-maintenance and fig4c-distance-largest-maintenance

> In the dPCA state space, the speed of the population trajectory was highest during encoding
> and lowest during maintenance, lower than in each other trial period, including fixation.

> The pairwise distance between the trajectories of different set sizes was larger during
> maintenance than during encoding, and the largest of the trial periods.

In [ ]:
rng = np.random.default_rng(512)
PER = {"fixation": (-6, -5), "encoding": (-5, -3), "maintenance": (-3, 0), "retrieval_a": (0, 1), "retrieval_b": (0, 2)}


def speed_dist(comps):
    Z = np.array([[c["z"][s] for c in comps[:15]] for s in range(3)])     # set x comp x time
    V = np.sqrt((np.diff(Z, axis=2) ** 2).sum(1)) / 15 / 0.02            # set x (time - 1)
    Dd = np.mean([np.sqrt(((Z[i] - Z[j]) ** 2).sum(0)) for i, j in [(0, 1), (0, 2), (1, 2)]], axis=0)
    out = {}
    for k, (a, b) in PER.items():
        m = (TT >= a) & (TT < b)
        out["V_" + k] = V.mean(0)[m[1:]].mean()
        out["D_" + k] = Dd[m].mean()
    return out


runs_sd = []
for r_ in range(2000):
    sub = rng.integers(0, len(pop), int(round(0.2 * len(pop))))
    runs_sd.append(speed_dist(S.dpca(XZ["a"][sub], 15, 0.0)[0]))
RS = pd.DataFrame(runs_sd)
for n_runs in (200, 2000):
    Rk = RS.iloc[:n_runs]
    print(f"== {n_runs} runs; per-period means (speed in z per s, distance in z):")
    print(Rk.mean().round(4).to_string())
    for kind in ("V", "D"):
        for ret in ("retrieval_a", "retrieval_b"):
            others = ["fixation", "encoding", ret]
            res = []
            for o in others:
                t, p = S.perm_t(Rk[f"{kind}_maintenance"], Rk[f"{kind}_{o}"], n_perm=2000, rng=rng)
                res.append(f"{o}: t = {t:.2f}, P = {p:.4f}")
            vals = {o: Rk[f"{kind}_{o}"].mean() for o in ["maintenance"] + others}
            print(f"   {'speed' if kind == 'V' else 'distance'}, {ret}: maintenance against " + "; ".join(res)
                  + f"; highest period {max(vals, key=vals.get)}, lowest {min(vals, key=vals.get)}")
full = speed_dist(DP[("a", 0.0)])
print("full population (no subsampling):", {k: round(v, 4) for k, v in full.items()})

**Result: Same outcome as reading (a), reproduced.** The distance between set-size trajectories is largest during maintenance and exceeds encoding with 2000 runs (7.61 against 7.41, t = 5.42, P = 0.0005) and with 200 runs (7.62 against 7.38, t = 2.05, P = 0.04).

**Result: Same outcome as reading (a), partially reproduced.** Speed is lowest during maintenance (0.75 z/s with 200 runs, below fixation 1.07, encoding 0.92 and retrieval 1.41 or 1.27; P = 0.0005 for each), but it is highest during retrieval, not encoding.

### figs3d-hippocampus-loadings

> Ranking the first 200 neurons of each region by their dPCA loading, hippocampal neurons had
> higher loadings than entorhinal and amygdala neurons at each rank except the first three,
> where entorhinal neurons had the highest loading.

In [ ]:
regs_pop = UN.region_atlas_b.to_numpy()[pop]
for comp_name, ci in [("(a) dPC2", 1), ("(b) dPC3", 2)]:
    w = DP[("a", 0.0)][ci]["decoder"]
    for lname, f_ in [("(i) absolute", np.abs), ("(ii) signed", lambda x: x)]:
        top = {r: np.sort(f_(w[regs_pop == r]))[::-1][:200] for r in REG}
        wt = stats.wilcoxon(top["hippocampus"], top["entorhinal"])
        print(f"{comp_name} [{DP[('a', 0.0)][ci]['marg']}], {lname}: hippocampus above entorhinal at {int(np.sum(top['hippocampus'] > top['entorhinal']))} of 200 ranks "
              f"(Wilcoxon P = {wt.pvalue:.2g}), above amygdala at {int(np.sum(top['hippocampus'] > top['amygdala']))}; "
              f"ranks 1-3 hipp {np.round(top['hippocampus'][:3], 4).tolist()} ent {np.round(top['entorhinal'][:3], 4).tolist()}")

**Result: Same outcome as reading (a).** For dPC2 the hippocampal top-200 loadings exceed the entorhinal ones at all 200 ranks under both loadings (Wilcoxon P = 1.4e-34) and the amygdalar ones at all ranks; for dPC3 at 197 and 200 ranks. Entorhinal neurons are not highest at the first three ranks in any reading.

## Summary

| Claim | Outcome under reading (b) |
|---|---|
| fig4e-maintenance-decodes-workload | Same as reading (a), reproduced |
| fig4d-encoding-decodes-correct | Same as reading (a), not reproduced |
| fig4e-workload-decoding-leave-one-subject-out | Same as reading (a), reproduced |
| fig4e-maintenance-neurons-carry-workload | Same as reading (a), reproduced |
| fig4e-no-decoding-6-vs-8 | Same as reading (a), reproduced |
| fig4f-retrieval-decodes-response | Same as reading (a), not reproduced |
| fig4a-dpca-workload-separation | Same as reading (a), not reproduced |
| fig4b-speed-lowest-maintenance | Same as reading (a), partially reproduced |
| fig4c-distance-largest-maintenance | Same as reading (a), reproduced |
| figs3d-hippocampus-loadings | Same as reading (a), reproduced |

The outcomes are judged in the first notebook and the attempt record. A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Differences from the paper's method are those of the first notebook (100 resampling runs and 250 null runs for decoding, 20 ms bins and no cross-validated regularization for dPCA). Reading (b) leaves out the 25 units (subject 7 PHL, 19; subject 9 PHL, 6) whose tip contact was labelled by hand, because their Brainnetome label is in neither deposit. What was found about the dataset: the flag of manually entered labels is in the G-Node deposit but not in the dandiset.
